In [1]:
import numpy as np
import torch

In [2]:
class SGD:
    def __init__(self, lr: float = 1e-3, beta1: float = 0.9, weight_decay: float = 0):
        self.lr = lr
        self.beta1 = beta1
        self.weight_decay = weight_decay

        self.m = []

    def step(self, params: list[np.ndarray], grads: list[np.ndarray]) -> list[np.ndarray]:
        for i, (param, grad) in enumerate(zip(params, grads)):
            if len(self.m) != len(params):
                self.m.append(np.zeros_like(param))

            m = self.m[i]
            m = self.beta1 * m + grad

            param_new = (param - self.lr * m - self.lr * self.weight_decay * param)
            self.m[i] = m
            params[i] = param_new
            
        return params

In [ ]:
class RMSprop:
    def __init__(self, lr: float = 1e-3, beta2: float = 0.999, eps: float = 1e-8, weight_decay: float = 0):
        self.lr = lr
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay

        self.v = []

    def step(self, params: list[np.ndarray], grads: list[np.ndarray]) -> list[np.ndarray]:
        for i, (param, grad) in enumerate(zip(params, grads)):
            if len(self.v) != len(params):
                self.v.append(np.zeros_like(param))

            v = self.v[i]
            v = self.beta2 * v + (1 - self.beta2) * np.square(grad)

            lr_step = self.lr / (np.sqrt(v) + self.eps)
            param_new = (param - lr_step * grad - self.lr * self.weight_decay * param)

            self.v[i] = v
            params[i] = param_new

        return params


In [5]:
class AdamW:

    def __init__(self, lr: float = 1e-3, beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8, weight_decay: float = 1e-2):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps 
        self.weight_decay = weight_decay
        self.t = 0

        self.m = []
        self.v = []        
    
    def step(self, params: list[np.ndarray], grads: list[np.ndarray]) -> list[np.ndarray]:
        self.t += 1

        for i, (param, grad) in enumerate(zip(params, grads)):
            if len(self.v) != len(params):
                self.m.append(np.zeros_like(param))
                self.v.append(np.zeros_like(param))

            m = self.m[i]
            v = self.v[i]

            m = self.beta1 * m + (1- self.beta1) * grad
            v = self.beta2 * v + (1 - self.beta2) * np.square(grad)


            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)

            adam_step = m_hat / (np.sqrt(v_hat) + self.eps)
            param_new = param - self.lr * adam_step - self.lr * self.weight_decay * param

            self.v[i] = v
            self.m[i] = m
            params[i] = param_new

        return params